# Symptom Annotation Tool

This notebook allows you to randomly sample 17 cases and annotate them for symptoms.
Progress is saved automatically to `gold_standard_trinary.csv`.

In [2]:
import pandas as pd
import os
import random
import glob
import ipywidgets as widgets
from IPython.display import display, clear_output

# --- CONFIGURATION ---
DATA_DIR = r"..\Clean Transcripts"
OUTPUT_FILE = "gold_standard_trinary.csv"

# Target counts for sampling
TARGETS = {
    "RES": 4,  # Respiratory
    "CAR": 4,  # Cardiac
    "MSK": 4,  # Musculoskeletal
    "GAS": 4,  # Gastrointestinal
    "DER": 4   # Dermatological (Note: Code will take max available if < 4)
}

# Structured List of Symptoms
SYMPTOM_GROUPS = {
    "Respiratory (The 'Outbreak' Core)": {
        "Fever": 'Includes: chills, night sweats, "burning up", "hot"',
        "Cough": 'Includes: hacking, productive/wet, dry, "barking"',
        "Dyspnea": 'Includes: shortness of breath, "winded", difficulty breathing',
        "Wheezing": 'Includes: whistling sound when breathing',
        "Rhinorrhea": 'Includes: runny nose, nasal drip',
        "Pharyngitis": 'Includes: sore throat, "scratchy throat',
        "Congestion": "stuffy nose"
    },
    "Cardiac": {
        "Chest Pain": 'Includes: tightness, pressure, "elephant on chest"',
        "Palpitations": 'Includes: racing heart, skipped beats, "thumping"',
        "Syncope": 'Includes: fainting, blacking out',
        "Edema": 'Includes: swollen ankles, leg swelling',
        "Fatigue":""
    },
    "Musculoskeletal": {
        "Myalgia": 'Includes: muscle aches, body aches',
        "Arthralgia": 'Includes: joint pain, "stiff joints"',
        "Back Pain": 'Includes: lower back pain, lumbar strain'
    },
    "Gastrointestinal": {
        "Nausea/Emesis": 'Includes: feeling sick to stomach, vomiting',
        "Diarrhea": 'Includes: loose stools',
        "Abdominal Pain": 'Includes: stomach cramps, "belly ache"'
    },
    "Dermatology": {
        "Pruritus": 'Includes: itching, "itchy skin"',
        "Rash": 'Includes: hives, redness, skin lesion, "breakout"'
    },
     "Additional": {
        "Headache": "",
        "Exposure To Sick":"",
        "Anosmia":"loss of sense of smell",
        "Ageusia":"lose of sense of taste",
    }
}

# Flattened list for DataFrame columns
ALL_SYMPTOMS = []
for category, symptoms in SYMPTOM_GROUPS.items():
    for symptom in symptoms:
        ALL_SYMPTOMS.append(symptom)

print("Setup complete.")

Setup complete.


In [3]:
def get_sampled_cases(data_dir, targets, output_file):
    # 1. If output file exists, load it to preserve existing samples/annotations
    if os.path.exists(output_file):
        print(f"Loading existing progress from {output_file}...")
        df = pd.read_csv(output_file)
        # Ensure all symptom columns exist
        for s in ALL_SYMPTOMS:
            if s not in df.columns:
                df[s] = 0 # Default to 0 (Not Present)
        return df

    # 2. Otherwise, perform new random sampling
    print("No existing file found. Performing new random sampling...")
    sampled_files = []
    
    all_files = glob.glob(os.path.join(data_dir, "*.txt"))
    file_map = {}
    
    # Group files by prefix
    for f in all_files:
        basename = os.path.basename(f)
        prefix = basename[:3]
        if prefix not in file_map:
            file_map[prefix] = []
        file_map[prefix].append(f)
        
    # Sample according to targets
    records = []
    for prefix, count in targets.items():
        available = file_map.get(prefix, [])
        if len(available) < count:
            print(f"Warning: Requested {count} for {prefix}, but only found {len(available)}. Taking all.")
            selection = available
        else:
            selection = random.sample(available, count)
            
        for path in selection:
            filename = os.path.basename(path)
            records.append({
                "filename": filename,
                "category": prefix,
                "full_path": path
            })
            
    # Create DataFrame
    df = pd.DataFrame(records)
    # Initialize symptoms columns with 0 (Not Present)
    for s in ALL_SYMPTOMS:
        df[s] = 0
        
    # Save initial version
    df.to_csv(output_file, index=False)
    print(f"Sampled {len(df)} cases. Initialized {output_file}.")
    return df

In [4]:
class AnnotationWidget:
    def __init__(self, df, output_file):
        self.df = df
        self.output_file = output_file
        self.index = 0
        self.total_cases = len(df)
        
        # UI Elements
        self.output_area = widgets.Output()
        self.transcript_area = widgets.Textarea(
            disabled=True,
            layout=widgets.Layout(width='50%', height='600px')
        )
        
        self.symptom_widgets = {}
        
        # Create Accordion for Symptom Groups
        accordion_children = []
        accordion_titles = []
        
        for group_name, symptoms_dict in SYMPTOM_GROUPS.items():
            group_widgets = []
            for s, description in symptoms_dict.items():
                # Label and Description
                lbl = widgets.HTML(f"<b>{s}</b><br><small style='color:gray'>{description}</small>")
                
                # Selection Buttons
                # Using ToggleButtons for clear, clickable options
                # 0: Not Present, 1: Negated, 2: Present
                w = widgets.ToggleButtons(
                    options=[('Not Present', 0), ('Negated', 1), ('Present', 2)],
                    description='', # Label is above
                    style={'button_width': '80px'},
                    layout=widgets.Layout(margin='2px')
                )
                
                # Update value on change
                w.observe(lambda change, s=s: self.update_value(s, change['new']), names='value')
                self.symptom_widgets[s] = w
                
                # Row Layout
                row = widgets.VBox([lbl, w], layout=widgets.Layout(margin='5px 0px 10px 0px'))
                group_widgets.append(row)
            
            accordion_children.append(widgets.VBox(group_widgets, layout=widgets.Layout(padding='10px')))
            accordion_titles.append(group_name)
            
        self.accordion = widgets.Accordion(children=accordion_children)
        for i, title in enumerate(accordion_titles):
            self.accordion.set_title(i, title)
            
        self.controls_area = widgets.VBox(
            [self.accordion],
            layout=widgets.Layout(width='48%', height='600px', overflow='auto')
        )
        
        # Navigation
        self.btn_prev = widgets.Button(description="<< Previous", layout=widgets.Layout(width='100px'))
        self.btn_next = widgets.Button(description="Next >>", layout=widgets.Layout(width='100px'))
        self.btn_save = widgets.Button(description="Force Save", button_style='success', layout=widgets.Layout(width='100px'))
        
        self.btn_prev.on_click(self.on_prev)
        self.btn_next.on_click(self.on_next)
        self.btn_save.on_click(self.save_csv)
        
        self.nav_box = widgets.HBox([self.btn_prev, self.btn_next, self.btn_save], layout=widgets.Layout(justify_content='center', margin='10px'))
        self.info_lbl = widgets.HTML(layout=widgets.Layout(margin='0px 0px 10px 0px'))
        
        self.main_layout = widgets.VBox([
            self.info_lbl,
            widgets.HBox([self.transcript_area, self.controls_area]),
            self.nav_box,
            self.output_area
        ])
        
        self.load_case()

    def load_case(self):
        row = self.df.iloc[self.index]
        filename = row['filename']
        category = row['category']
        
        # Try to locate full path (reconstruct if needed)
        if 'full_path' in row and os.path.exists(row['full_path']):
            path = row['full_path']
        else:
            # Fallback reconstruction
            path = os.path.join(DATA_DIR, filename)
            
        content = f"[File not found: {path}]"
        if os.path.exists(path):
            with open(path, 'r', encoding='utf-8') as f:
                content = f.read()
        
        self.transcript_area.value = content
        self.info_lbl.value = f"<h2>Case {self.index + 1} / {self.total_cases}: {filename} ({category})</h2>"
        
        # Set widget values
        for s, w in self.symptom_widgets.items():
            val = row.get(s, 0)
            if pd.isna(val):
                val = 0
            w.value = int(val)

    def update_value(self, symptom, value):
        self.df.at[self.index, symptom] = value
        
    def save_csv(self, _=None):
        self.df.to_csv(self.output_file, index=False)
        with self.output_area:
            clear_output(wait=True)
            print(f"Saved to {self.output_file}")

    def on_prev(self, b):
        self.save_csv()
        if self.index > 0:
            self.index -= 1
            self.load_case()

    def on_next(self, b):
        self.save_csv()
        if self.index < self.total_cases - 1:
            self.index += 1
            self.load_case()

    def display(self):
        display(self.main_layout)

In [5]:
# Run the Tool
df_cases = get_sampled_cases(DATA_DIR, TARGETS, OUTPUT_FILE)
app = AnnotationWidget(df_cases, OUTPUT_FILE)
app.display()

Loading existing progress from gold_standard_trinary.csv...


In [60]:
outputtedresults = pd.read_csv("gold_standard_trinary.csv")
print(outputtedresults)

       filename category                                          full_path  \
0   RES0123.txt      RES  C:\Users\arjun\OneDrive\Desktop\Symptom Detect...   
1   RES0046.txt      RES  C:\Users\arjun\OneDrive\Desktop\Symptom Detect...   
2   RES0214.txt      RES  C:\Users\arjun\OneDrive\Desktop\Symptom Detect...   
3   RES0167.txt      RES  C:\Users\arjun\OneDrive\Desktop\Symptom Detect...   
4   CAR0003.txt      CAR  C:\Users\arjun\OneDrive\Desktop\Symptom Detect...   
5   CAR0001.txt      CAR  C:\Users\arjun\OneDrive\Desktop\Symptom Detect...   
6   CAR0002.txt      CAR  C:\Users\arjun\OneDrive\Desktop\Symptom Detect...   
7   CAR0004.txt      CAR  C:\Users\arjun\OneDrive\Desktop\Symptom Detect...   
8   MSK0024.txt      MSK  C:\Users\arjun\OneDrive\Desktop\Symptom Detect...   
9   MSK0044.txt      MSK  C:\Users\arjun\OneDrive\Desktop\Symptom Detect...   
10  MSK0033.txt      MSK  C:\Users\arjun\OneDrive\Desktop\Symptom Detect...   
11  MSK0029.txt      MSK  C:\Users\arjun\OneDrive\De

In [61]:
outputtedresults.to_csv("gold_standard_trinary_backup")